# Fault analysis

## Imports and definitions

In [ ]:
import itertools

import stim
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch
from utils.circuit_faults import analyse_heuristic

In [ ]:
def append_mpp(circuitry: Circuitry, steane: SteaneCodePatch, stage: str):
    for label, stabilizer in steane.pauli_stabilizers.items():
        circuitry.append("MPP", stim.PauliString(stabilizer))
        qubits.record_measurement(f"{stage}_{label}")
        circuitry.append_tick()
    circuitry.append("MPP", stim.PauliString(steane.logical(Pauli.Y)))
    qubits.record_measurement(f"{stage}_YL")
    circuitry.append_tick()

## Analysis of Injection stage

In [ ]:
qubits: QubitArray = QubitArray(dimensions=(5,5))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(array=qubits)

circuitry.annotate_polygons(steane.get_polygons())

steane.append_preparation(circuitry)
append_mpp(circuitry, steane, "MEASUREMENT")

# Annotate the X&Z Steane stabilizer detectors
for label in steane.pauli_stabilizers.keys():
    circuitry.annotate_detector(f"MEASUREMENT_{label}")

# Attach the Y_L observable
circuitry.attach_observable(0, "MEASUREMENT_YL")

circuitry.detectors_report(open_boundaries=True)

display(Markdown(f"[Open in Crumble (Injection)]({circuitry.to_crumble_url()})"))

In [ ]:
circuit = circuitry.as_stim(per=0.01, noiseless_measurement_ticks=7)
analyse_heuristic(circuit)

## Analysis of Superdense Code Cycle

In [ ]:
qubits: QubitArray = QubitArray(dimensions=(5,5))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(array=qubits)

circuitry.annotate_polygons(steane.get_polygons())

append_mpp(circuitry, steane, "PREPARATION")
steane.append_superdense_cycle(circuitry, prefix="STN:SDC0")
append_mpp(circuitry, steane, "MEASUREMENT")

# Annotate the X&Z preparation detectors and the Z measurement detectors
for label in steane.pauli_stabilizers.keys():
    circuitry.annotate_detector(f"PREPARATION_{label}", f"STN:SDC0:{label}")
    if label[0] == 'Z':
        circuitry.annotate_detector(f"MEASUREMENT_{label}", f"STN:SDC0:{label}")

# Annotate the X measurement detectors
circuitry.annotate_detector(f"MEASUREMENT_XB", f"STN:SDC0:XG", )
circuitry.annotate_detector(f"MEASUREMENT_XG", *(f"STN:SDC0:{label}" for label in ["XG", "XR"]))
circuitry.annotate_detector(f"MEASUREMENT_XR")

# Attach the Y_L observable
circuitry.attach_observable(0, "MEASUREMENT_YL", "PREPARATION_YL", "STN:SDC0:XB", "STN:SDC0:XR")

circuitry.detectors_report(open_boundaries=True)

display(Markdown(f"[Open in Crumble (Superdense)]({circuitry.to_crumble_url()})"))

In [ ]:
circuit = circuitry.as_stim(per=0.01, noiseless_preparation_ticks=7, noiseless_measurement_ticks=7)
analyse_heuristic(circuit)

## Analysis of Cultivation stage

In [ ]:
qubits: QubitArray = QubitArray(dimensions=(5,5))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(array=qubits)

circuitry.annotate_polygons(steane.get_polygons())

append_mpp(circuitry, steane, "PREPARATION")
steane.append_cultivation(circuitry)
append_mpp(circuitry, steane, "MEASUREMENT")

# Annotate the X&Z preparation&measurement detectors
for label in steane.pauli_stabilizers.keys():
    circuitry.annotate_detector(f"PREPARATION_{label}", f"MEASUREMENT_{label}")

# Annotate the X detectors
for number in range(2, 7):
    circuitry.annotate_detector(f"STN:CULT:X{number}")

# Annotate the Y measurement detector
circuitry.annotate_detector("PREPARATION_YL", "STN:CULT:X0")
circuitry.annotate_detector("STN:CULT:X0", "STN:CULT:X1")

# Attach the Y_L observable
circuitry.attach_observable(0, "MEASUREMENT_YL", *(f"STN:CULT:X{m}" for m in [1,2,3,5]))

display(Markdown(f"[Open in Crumble (Cultivation)]({circuitry.to_crumble_url()})"))

circuitry.detectors_report(open_boundaries=True)

In [ ]:
circuit = circuitry.as_stim(per=0.01, noiseless_preparation_ticks=7, noiseless_measurement_ticks=7)
analyse_heuristic(circuit)

## Analysis of Teleportation stage

In [ ]:
TELEPORTATION_ROUNDS = 3
qubits: QubitArray = QubitArray(dimensions=(5,5))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(array=qubits)

circuitry.annotate_polygons(steane.get_polygons())

append_mpp(circuitry, steane, "PREPARATION")
for rnd in range(TELEPORTATION_ROUNDS):
    steane.append_teleportation(circuitry, prefix=f"STN:TPT{rnd}")
append_mpp(circuitry, steane, "MEASUREMENT")

# Annotate the X&Z preparation detectors and the Z measurement detectors
last = TELEPORTATION_ROUNDS-1
for label in filter(lambda lb: lb != 'XB', steane.pauli_stabilizers.keys()):
    circuitry.annotate_detector(f"PREPARATION_{label}", f"STN:TPT0:{label}")
    if label[0] == 'Z' or label == 'XR':
        circuitry.annotate_detector(f"MEASUREMENT_{label}", f"STN:TPT{last}:{label}")
    elif label == 'XG':
        circuitry.annotate_detector(f"MEASUREMENT_{label}", f"STN:TPT{last}:{label}", f"STN:TPT{last}:XR")


# Annotate the XB detectors
circuitry.annotate_detector(
    "PREPARATION_XB", "MEASUREMENT_XB",
    *(f"STN:TPT{rnd}:XB" for rnd in range(TELEPORTATION_ROUNDS)),
    *(f"STN:TPT{rnd}:XG" for rnd in range(TELEPORTATION_ROUNDS))
)

for xbs in itertools.combinations(range(TELEPORTATION_ROUNDS), 2):
    circuitry.annotate_detector(
        "PREPARATION_XB", "MEASUREMENT_XB",
        *(f"STN:TPT{rnd}:XG" for rnd in range(TELEPORTATION_ROUNDS)),
        *(f"STN:TPT{rnd}:XB" for rnd in xbs)
    )

for label, (prev, curr) in itertools.product(steane.pauli_stabilizers.keys(), itertools.pairwise(range(TELEPORTATION_ROUNDS))):
    match label[0]:
        case 'X':
            if label == 'XG':
                circuitry.annotate_detector(f"STN:TPT{curr}:XG", f"STN:TPT{prev}:XG", f"STN:TPT{prev}:XR")
            elif label == 'XR':
                circuitry.annotate_detector(f"STN:TPT{curr}:XR")
        case 'Z':
            circuitry.annotate_detector(f"STN:TPT{curr}:{label}", f"STN:TPT{prev}:{label}")

# Attach the Y_L observable
circuitry.attach_observable(
    0, "PREPARATION_YL", "MEASUREMENT_YL",
    *(f"STN:TPT{rnd}:{ma}" for rnd, ma in itertools.product(range(TELEPORTATION_ROUNDS), ["XR", "XB"]))
)

display(Markdown(f"[Open in Crumble (Teleportation)]({circuitry.to_crumble_url()})"))

circuitry.detectors_report(open_boundaries=True)

In [ ]:
circuit = circuitry.as_stim(per=0.01, noiseless_preparation_ticks=7, noiseless_measurement_ticks=7)
analyse_heuristic(circuit)